# Feature Engineering

Create model-ready features from the cleaned air quality data.

In [1]:
import sqlite3
import numpy as np
import pandas as pd

con = sqlite3.connect("../data/airlens.db")

df = pd.read_sql_query(
    "SELECT * FROM air_quality_validated",
    con
)

con.close()

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(["City", "Date"]).reset_index(drop=True)

# Build temporal, rolling, or domain-specific features here.

In [2]:
city_quality = df.groupby("City").agg(
    total_records=("Date", "count"),
    valid_aqi=("Valid_AQI", "count")
)

city_quality["coverage"] = (
    city_quality["valid_aqi"] /
    city_quality["total_records"] * 100
)

reliable_cities = city_quality[
    (city_quality["valid_aqi"] >= 500) &
    (city_quality["coverage"] >= 80)
].index.tolist()

ml_df = df[df["City"].isin(reliable_cities)].copy()

ml_df.shape

(18395, 15)

In [4]:
ml_df.head()

,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,AQI,AQI_Bucket,Valid_AQI,AQI_Outlier
2122,Amaravati,2017-11-24,71.36,115.75,1.75,20.65,12.40,12.19,0.10,10.76,109.26,NaN,NaN,NaN,0
2123,Amaravati,2017-11-25,81.40,124.50,1.44,20.50,12.08,10.72,0.12,15.24,127.09,184.0,Moderate,184.0,0
2124,Amaravati,2017-11-26,78.32,129.06,1.26,26.00,14.85,10.28,0.14,26.96,117.44,197.0,Moderate,197.0,0
2125,Amaravati,2017-11-27,88.76,135.32,6.60,30.85,21.77,12.91,0.11,33.59,111.81,198.0,Moderate,198.0,0
2126,Amaravati,2017-11-28,64.18,104.09,2.56,28.07,17.01,11.42,0.09,19.00,138.18,188.0,Moderate,188.0,0


In [5]:
ml_df["Year"] = ml_df["Date"].dt.year
ml_df["Month"] = ml_df["Date"].dt.month
ml_df["DayOfWeek"] = ml_df["Date"].dt.dayofweek
ml_df["DayOfYear"] = ml_df["Date"].dt.dayofyear

In [6]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    return "Post-Monsoon"

ml_df["Season"] = ml_df["Month"].apply(get_season)

In [7]:
ml_df["Target_AQI"] = (
    ml_df.groupby("City")["Valid_AQI"]
    .shift(-1)
)

In [8]:
ml_df[
    ["City", "Date", "Valid_AQI", "Target_AQI"]
].head(10)

,City,Date,Valid_AQI,Target_AQI
2122,Amaravati,2017-11-24,NaN,184.0
2123,Amaravati,2017-11-25,184.0,197.0
2124,Amaravati,2017-11-26,197.0,198.0
2125,Amaravati,2017-11-27,198.0,188.0
2126,Amaravati,2017-11-28,188.0,173.0
2127,Amaravati,2017-11-29,173.0,165.0
2128,Amaravati,2017-11-30,165.0,191.0
2129,Amaravati,2017-12-01,191.0,191.0
2130,Amaravati,2017-12-02,191.0,227.0
2131,Amaravati,2017-12-03,227.0,168.0


In [9]:
g = ml_df.groupby("City")

ml_df["AQI_Lag1"] = g["Valid_AQI"].shift(1)
ml_df["AQI_Lag2"] = g["Valid_AQI"].shift(2)
ml_df["AQI_Lag3"] = g["Valid_AQI"].shift(3)
ml_df["AQI_Lag7"] = g["Valid_AQI"].shift(7)

In [10]:
ml_df["AQI_Roll3"] = (
    g["Valid_AQI"]
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

ml_df["AQI_Roll7"] = (
    g["Valid_AQI"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [11]:
for col in ["PM2.5", "PM10", "NO2", "CO", "SO2", "O3"]:
    ml_df[f"{col}_Lag1"] = (
        ml_df.groupby("City")[col].shift(1)
    )

In [12]:
ml_df["AQI_Change1"] = (
    ml_df["Valid_AQI"] - ml_df["AQI_Lag1"]
)

ml_df["PM25_Change1"] = (
    ml_df["PM2.5"] - ml_df["PM2.5_Lag1"]
)

In [13]:
ml_df.columns

Index(['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2',
       'O3', 'AQI', 'AQI_Bucket', 'Valid_AQI', 'AQI_Outlier', 'Year', 'Month',
       'DayOfWeek', 'DayOfYear', 'Season', 'Target_AQI', 'AQI_Lag1',
       'AQI_Lag2', 'AQI_Lag3', 'AQI_Lag7', 'AQI_Roll3', 'AQI_Roll7',
       'PM2.5_Lag1', 'PM10_Lag1', 'NO2_Lag1', 'CO_Lag1', 'SO2_Lag1', 'O3_Lag1',
       'AQI_Change1', 'PM25_Change1'],
      dtype='str')

In [14]:
ml_df[
    [
        "City",
        "Date",
        "Valid_AQI",
        "AQI_Lag1",
        "AQI_Lag7",
        "AQI_Roll7",
        "PM2.5",
        "PM2.5_Lag1",
        "Target_AQI"
    ]
].head(15)

,City,Date,Valid_AQI,AQI_Lag1,AQI_Lag7,AQI_Roll7,PM2.5,PM2.5_Lag1,Target_AQI
2122,Amaravati,2017-11-24,NaN,NaN,NaN,NaN,71.36,NaN,184.0
2123,Amaravati,2017-11-25,184.0,NaN,NaN,NaN,81.40,71.36,197.0
2124,Amaravati,2017-11-26,197.0,184.0,NaN,NaN,78.32,81.40,198.0
2125,Amaravati,2017-11-27,198.0,197.0,NaN,NaN,88.76,78.32,188.0
2126,Amaravati,2017-11-28,188.0,198.0,NaN,NaN,64.18,88.76,173.0
2127,Amaravati,2017-11-29,173.0,188.0,NaN,NaN,72.47,64.18,165.0
2128,Amaravati,2017-11-30,165.0,173.0,NaN,NaN,69.80,72.47,191.0
2129,Amaravati,2017-12-01,191.0,165.0,NaN,NaN,73.96,69.80,191.0
2130,Amaravati,2017-12-02,191.0,191.0,184.0,185.142857,89.90,73.96,227.0
2131,Amaravati,2017-12-03,227.0,191.0,197.0,186.142857,87.14,89.90,168.0


In [15]:
ml_df["Month_Sin"] = np.sin(2 * np.pi * ml_df["Month"] / 12)
ml_df["Month_Cos"] = np.cos(2 * np.pi * ml_df["Month"] / 12)

ml_df["DayOfYear_Sin"] = np.sin(
    2 * np.pi * ml_df["DayOfYear"] / 365.25
)

ml_df["DayOfYear_Cos"] = np.cos(
    2 * np.pi * ml_df["DayOfYear"] / 365.25
)

In [16]:
ml_df = ml_df.dropna(subset=["Target_AQI"]).copy()

In [17]:
ml_df.shape

(16963, 39)

In [18]:
ml_df["Target_AQI"].isna().sum()

np.int64(0)

In [19]:
num_features = [
    "Valid_AQI",
    "PM2.5",
    "PM10",
    "NO2",
    "CO",
    "SO2",
    "O3",

    "AQI_Lag1",
    "AQI_Lag2",
    "AQI_Lag3",
    "AQI_Lag7",
    "AQI_Roll3",
    "AQI_Roll7",

    "PM2.5_Lag1",
    "PM10_Lag1",
    "NO2_Lag1",
    "CO_Lag1",
    "SO2_Lag1",
    "O3_Lag1",

    "AQI_Change1",
    "PM25_Change1",

    "Month_Sin",
    "Month_Cos",
    "DayOfYear_Sin",
    "DayOfYear_Cos"
]

cat_features = [
    "City",
    "Season",
    "DayOfWeek"
]

In [20]:
train_df = ml_df[
    ml_df["Date"] < "2019-01-01"
].copy()

val_df = ml_df[
    (ml_df["Date"] >= "2019-01-01") &
    (ml_df["Date"] < "2020-01-01")
].copy()

test_df = ml_df[
    ml_df["Date"] >= "2020-01-01"
].copy()

In [21]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (10524, 39)
Validation: (4269, 39)
Test: (2170, 39)


In [22]:
print(
    "Train:",
    train_df["Date"].min(),
    "to",
    train_df["Date"].max()
)

print(
    "Validation:",
    val_df["Date"].min(),
    "to",
    val_df["Date"].max()
)

print(
    "Test:",
    test_df["Date"].min(),
    "to",
    test_df["Date"].max()
)

Train: 2015-01-01 00:00:00 to 2018-12-31 00:00:00
Validation: 2019-01-01 00:00:00 to 2019-12-31 00:00:00
Test: 2020-01-01 00:00:00 to 2020-06-30 00:00:00


In [23]:
print("Train cities:", train_df["City"].nunique())
print("Validation cities:", val_df["City"].nunique())
print("Test cities:", test_df["City"].nunique())

Train cities: 12
Validation cities: 12
Test cities: 12


In [24]:
ml_df.to_csv(
    "../data/processed/engineered_air_quality.csv",
    index=False
)